## Get Email Data

In [1]:
import warnings

import os
import sys
import json
import re
import time 
import random
import math
import torch
import pandas as pd
import numpy as np  

import matplotlib.pyplot as plt

from typing import List, Tuple
from sklearn.model_selection import train_test_split

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments
from transformers import DataCollatorForTokenClassification
from datasets import Dataset, DatasetDict
from evaluate import load
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning, module=".*torch.*")

In [2]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter


# Default path to the dataset
#default_path =  r'C:\Users\Piyush\.cache\kagglehub\datasets\wcukierski\enron-email-dataset\versions\2\'
# Set the path to the file you'd like to load
file_path = 'D:\Dev\GitHub\AI-ML-Course\Projects\External\EMail-Signature-Detection\data\enron_email_dataset.csv'
default_path =  'C:/Users/Piyush/.cache/kagglehub/datasets/wcukierski/enron-email-dataset/versions/2/'

# Download latest version
path = kagglehub.dataset_download("wcukierski/enron-email-dataset")

print("Path to dataset files:", path)

# Download the Enron emails dataset using kagglehub
#dataset = kagglehub.dataset_download(    
    #"wcukierski/enron-email-dataset",
    #file_path
#)

print("Dataset downloaded successfully.")
email_file_path =  'C:/Users/Piyush/.cache/kagglehub/datasets/wcukierski/enron-email-dataset/versions/2/emails.csv'

# Load the latest version
#data = kagglehub.load_dataset(
#  KaggleDatasetAdapter.PANDAS,
#  "wcukierski/enron-email-dataset",
#  email_file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
#)

data = pd.read_csv(email_file_path)
print("First 5 records:", data.head())

Path to dataset files: C:\Users\Piyush\.cache\kagglehub\datasets\wcukierski\enron-email-dataset\versions\2
Dataset downloaded successfully.
First 5 records:                        file                                            message
0     allen-p/_sent_mail/1.  Message-ID: <18782981.1075855378110.JavaMail.e...
1    allen-p/_sent_mail/10.  Message-ID: <15464986.1075855378456.JavaMail.e...
2   allen-p/_sent_mail/100.  Message-ID: <24216240.1075855687451.JavaMail.e...
3  allen-p/_sent_mail/1000.  Message-ID: <13505866.1075863688222.JavaMail.e...
4  allen-p/_sent_mail/1001.  Message-ID: <30922949.1075863688243.JavaMail.e...


In [3]:
data.info

<bound method DataFrame.info of                              file  \
0           allen-p/_sent_mail/1.   
1          allen-p/_sent_mail/10.   
2         allen-p/_sent_mail/100.   
3        allen-p/_sent_mail/1000.   
4        allen-p/_sent_mail/1001.   
...                           ...   
517396  zufferli-j/sent_items/95.   
517397  zufferli-j/sent_items/96.   
517398  zufferli-j/sent_items/97.   
517399  zufferli-j/sent_items/98.   
517400  zufferli-j/sent_items/99.   

                                                  message  
0       Message-ID: <18782981.1075855378110.JavaMail.e...  
1       Message-ID: <15464986.1075855378456.JavaMail.e...  
2       Message-ID: <24216240.1075855687451.JavaMail.e...  
3       Message-ID: <13505866.1075863688222.JavaMail.e...  
4       Message-ID: <30922949.1075863688243.JavaMail.e...  
...                                                   ...  
517396  Message-ID: <26807948.1075842029936.JavaMail.e...  
517397  Message-ID: <25835861.1075842029959

In [4]:
#data.columns
data.describe()

,file,message
count,517401,517401
unique,517401,517401
top,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
freq,1,1


In [5]:
def clean_text(text):
    return re.sub(r'[\r\n\t]', ' ', text)

def is_missing_signature(content):
    return '@' not in content

def has_multiple_emails(content):
    return len(re.findall(r'@', content)) > 1


In [15]:
data.rename(columns={'file': 'message_id'}, inplace=True)

data.rename(columns={'message': 'content'}, inplace=True)
# Extract the 'message' column as a list
messages = data['content'].tolist()

# Initialize counters
multiple_at_count = 0
single_at_count = 0
missing_at_count = 0

# Scan each message
for message in messages:
    if has_multiple_emails(message):
        multiple_at_count += 1
    elif is_missing_signature(message):
        missing_at_count += 1
    else:
        single_at_count += 1

print(f"Messages with multiple '@' chars: {multiple_at_count}")
print(f"Messages with a single '@' char: {single_at_count}")
print(f"Messages missing '@' char: {missing_at_count}")

Messages with multiple '@' chars: 517401
Messages with a single '@' char: 0
Messages missing '@' char: 0


In [13]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Load tokenizer and model from the local directory
MODEL_NAME = "models/dslim-bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, local_files_only=True)

print("Model and tokenizer loaded successfully.")
LABELS = {"O": 0, "PER": 1, "EML": 2}
ID2LABEL = {v: k for k, v in LABELS.items()}


Model and tokenizer loaded successfully.


In [21]:
def extract_entities(text: str) -> List[Tuple[str, str]]:
    tokens = text.split()
    entities = []

    for token in tokens:
        if re.match(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", token):
            entities.append((token, "EML"))
        elif token.istitle():
            entities.append((token, "PER"))
        else:
            entities.append((token, "O"))
    return entities


def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128,
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


def prepare_dataset(clean_data_path: str):
    with open(clean_data_path, "r") as f:
        data = json.load(f)

    dataset = []
    for item in data:
        entities = extract_entities(item["content"])
        tokens = [t[0] for t in entities]
        ner_tags = [LABELS[t[1]] for t in entities]
        dataset.append({"tokens": tokens, "ner_tags": ner_tags})

    train_val, test = train_test_split(dataset, test_size=0.2, random_state=42)
    train, val = train_test_split(train_val, test_size=0.25, random_state=42)  # 60/20/20 split

    dataset_dict = DatasetDict({
        "train": Dataset.from_list(train),
        "validation": Dataset.from_list(val),
        "test": Dataset.from_list(test)
    })

    return dataset_dict.map(tokenize_and_align_labels, batched=True)


def train_and_evaluate(dataset_dict):
    args = TrainingArguments(
        output_dir="./ner_output",
        #evaluation_strategy="epoch",
        logging_dir="./logs",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        save_total_limit=1,
        logging_steps=10,
    )

    data_collator = DataCollatorForTokenClassification(tokenizer)
    metric = load("seqeval")

    def compute_metrics(p):
        predictions, labels = p
        predictions = torch.argmax(torch.tensor(predictions), axis=2)

        true_predictions = [
            [ID2LABEL[p] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]
        true_labels = [
            [ID2LABEL[l] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]
        results = metric.compute(predictions=true_predictions, references=true_labels)
        return {"precision": results["overall_precision"], "recall": results["overall_recall"], "f1": results["overall_f1"], "accuracy": results["overall_accuracy"]}

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset_dict["train"],
        eval_dataset=dataset_dict["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    # Evaluate on test set
    predictions, labels, _ = trainer.predict(dataset_dict["test"])
    predictions = torch.argmax(torch.tensor(predictions), axis=2)
    test_results = []
    for i, (pred, label_ids) in enumerate(zip(predictions, labels)):
        tokens = dataset_dict["test"][i]["tokens"]
        final_labels = [ID2LABEL[l] for l in label_ids if l != -100]
        final_preds = [ID2LABEL[p] for p, l in zip(pred, label_ids) if l != -100]

        name = next((tok for tok, tag in zip(tokens, final_preds) if tag == "PER"), None)
        email = next((tok for tok, tag in zip(tokens, final_preds) if tag == "EML"), None)
        test_results.append({
            "message_id": i,
            "name": name,
            "email": email
        })

    with open("data/test_results.json", "w") as f:
        json.dump(test_results, f, indent=2)

    # Count failures
    missing_name = sum(1 for r in test_results if not r["name"])
    missing_email = sum(1 for r in test_results if not r["email"])

    print(f"Missing name in {missing_name} messages")
    print(f"Missing email in {missing_email} messages")    

In [18]:
# Reduce the total rows to 1/5 of the original data
data = data.sample(frac=0.2, random_state=42).reset_index(drop=True)
# Convert the DataFrame to a list of dictionaries
data_dict = data.to_dict(orient="records")

# Write the data to a JSON file
with open("data/clean_data.json", "w") as json_file:
    json.dump(data_dict, json_file, indent=2)

print("Data has been written to data/clean_data.json")

Data has been written to data/clean_data.json


In [23]:

dataset_dict = prepare_dataset("data/clean_data.json")
train_and_evaluate(dataset_dict)

Map:   0%|          | 0/62088 [00:00<?, ? examples/s]

Map:   0%|          | 0/20696 [00:00<?, ? examples/s]

Map:   0%|          | 0/20696 [00:00<?, ? examples/s]

Step,Training Loss
10,1.217400
20,0.057700
30,0.040700
40,0.036100
50,0.044000
60,0.026100
70,0.024200
80,0.024300
90,0.023700
100,0.016100


KeyError: tensor(0)